# Can multi-stakeholder agents evaluate a therapeutic asset? 


## TRANSEURO Case: Agentic vs. Single-LLM Evidence Synthesis

**Research question:** Does agentic reasoning (independent tool-calling, iterative
retrieval) produce more thorough, better-grounded therapeutic risk assessments
than a single-call LLM reading a fixed evidence snapshot?

**Case:** TRANSEURO (NCT01898390), a redesigned fetal ventral mesencephalic
tissue transplant trial for Parkinson's disease. Decision point: **December 31,
2014**, just before grafting began (2015-2018), assessing whether the
redesign adequately addressed graft-induced dyskinesia (GID), the central
risk identified in earlier NIH-funded trials (Freed 2001, Olanow 2003).

This case was chosen deliberately as a **contamination-failed** case (see
project-wide contamination protocol): the model already has confident
recall of TRANSEURO's actual outcome, so this case is evaluated on
**reasoning quality** (citation fidelity, evidence grounding, calibration),
not outcome prediction.

## Corpus construction
- Three independently pre-specified PubMed queries ("angles"): program-specific
  (TRANSEURO/Barker), disease-modality general (fetal VM transplantation,
  1970-2014), and field-mechanism (graft-induced dyskinesia, 1990-2014)
- Date cutoff enforced **twice**: at query time (`[pdat]` filter) and again
  by re-checking each result's actual `pubdate` — the query-time filter alone
  let two post-cutoff papers through during testing
- Selection: union of top-10 by relevance rank and top-10 by citation count
  per angle (relevance alone buries older foundational papers; citation
  count alone under-weights recent ones)
- Access checked via PMC and Unpaywall; abstract-only where no open full
  text exists (logged explicitly, not silently substituted)
- No manual paper curation — automated, reproducible queries only, to avoid
  hindsight bias in evidence selection

## Four-condition design

The code and result filenames below use `week1`-`week4`, reflecting the order these
conditions were actually built in. The write-up and graphics use plainer labels for
the same conditions, mapped here so the two are easy to cross-reference:

| Code label | Personas | Evidence | Tool calls | Called in the write-up |
|---|---|---|---|---|
| `week1` | Single call, 4 sections | Frozen corpus | None | *(persona-separation check only, not featured in the write-up)* |
| `week2` | 4 independent calls | Frozen corpus | None | "No search" |
| `week3` | 4 independent calls | Frozen corpus | Capped at 1 round | "One search" |
| `week4` | 4 independent calls | Frozen corpus | Unlimited rounds | "Repeated / agentic search" |

Week 3→4 is the core "agentic vs. not" comparison; Week 1→2 tests whether
persona separation alone changes output, independent of tool access.

**Personas:** Biology, Clinical, Regulatory, CMC/Manufacturing (Commercial
dropped — TRANSEURO is an academic/EU-funded consortium trial, not a
commercially sponsored program).


## Setup

This notebook needs a `.env` file in the same directory, containing:

```
ANTHROPIC_API_KEY=your_key_here
NCBI_API_KEY=your_ncbi_key_here
NCBI_EMAIL=your_email@example.com
```

`ANTHROPIC_API_KEY` is required for the Week 1-4 persona calls. `NCBI_API_KEY` is optional but recommended (raises the PubMed rate limit from 3 to 10 requests/second, register free at https://www.ncbi.nlm.nih.gov/account/). `NCBI_EMAIL` is NCBI's requested etiquette identifier for API requests, not a credential.

Dependencies: `pip install requests pandas anthropic python-dotenv`


In [ ]:
import requests
import time
import xml.etree.ElementTree as ET
import pandas as pd
import os
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY, NCBI_API_KEY, NCBI_EMAIL from a local .env file

EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# Optional but recommended: register a free NCBI API key to raise the rate
# limit from 3 to 10 requests/second: https://www.ncbi.nlm.nih.gov/account/
NCBI_API_KEY = os.getenv("NCBI_API_KEY")  # set in your own .env file, or leave unset

# NCBI etiquette: identify yourself in requests
TOOL_NAME = "transeuro-corpus-poc"
EMAIL = os.getenv("NCBI_EMAIL", "your_email@example.com")  # set your own in .env


In [7]:
pd.set_option("display.max_colwidth", None)

## Pipeline for building PubMed corpus


esearch → esummary → enforce_date_cutoff →  pubtype classification → PMC check →  DOI fetch  → Unpaywall check →  citation counts fetch → select_by_either_metric → corpus: combine selected papers & remove duplicates → fetch abstracts for corpus papers

In [17]:
def esearch(query: str, retmax: int = 200, sort: str = "relevance") -> dict:
    """Run a PubMed search and return the raw esearch response (JSON).
    retmax is set high here (200) so we can inspect the full relevance
    distribution before deciding a cutoff, not to imply we'll use all 200.
    """
    params = {
        "db": "pubmed",
        "term": query,
        "retmode": "json",
        "retmax": retmax,
        "sort": sort,
        "tool": TOOL_NAME,
        "email": EMAIL,
    }
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    resp = requests.get(f"{EUTILS_BASE}/esearch.fcgi", params=params, timeout=15)
    resp.raise_for_status()
    return resp.json()


def esummary(pmids: list) -> pd.DataFrame:
    """Fetch title, journal, publication date, and publication type(s) for PMIDs."""
    if not pmids:
        return pd.DataFrame(columns=["pmid", "title", "journal", "pubdate", "pubtypes"])
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "json",
        "tool": TOOL_NAME,
        "email": EMAIL,
    }
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    resp = requests.get(f"{EUTILS_BASE}/esummary.fcgi", params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()["result"]
    rows = []
    for pmid in data.get("uids", []):
        rec = data[pmid]
        rows.append({
            "pmid": pmid,
            "title": rec.get("title", ""),
            "journal": rec.get("fulljournalname", ""),
            "pubdate": rec.get("pubdate", ""),
            "pubtypes": rec.get("pubtype", []),  # e.g. ["Journal Article", "Review"]
        })
    return pd.DataFrame(rows)

In [18]:
def enforce_date_cutoff(df: pd.DataFrame, cutoff_year: int = 2014) -> pd.DataFrame:
    """Strictly re-filter on the actual pubdate field, never trust PubMed's
    server-side date filter alone for something this important."""
    df = df.copy()
    df["pub_year"] = df["pubdate"].str.extract(r"(\d{4})").astype(float)
    before = len(df)
    df = df[df["pub_year"] <= cutoff_year].reset_index(drop=True)
    dropped = before - len(df)
    if dropped > 0:
        print(f"Dropped {dropped} paper(s) with pub_year > {cutoff_year}, "
              f"despite passing the API's own date filter.")
    return df


In [19]:
def check_pmc_fulltext(pmid: str) -> str | None:
    """Return the PMCID (e.g. 'PMC1234567') if this PMID has a PMC full-text
    record, otherwise None. Uses XML retmode, elink's native format."""
    params = {
        "dbfrom": "pubmed",
        "db": "pmc",
        "id": pmid,
        "linkname": "pubmed_pmc",
        "tool": TOOL_NAME,
        "email": EMAIL,
    }
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    resp = requests.get(f"{EUTILS_BASE}/elink.fcgi", params=params, timeout=15)
    resp.raise_for_status()

    try:
        root = ET.fromstring(resp.content)
    except ET.ParseError as e:
        print(f"XML parse error for PMID {pmid}: {e}")
        print(resp.text[:300])
        return None

    for linksetdb in root.findall(".//LinkSetDb"):
        if linksetdb.findtext("LinkName") == "pubmed_pmc":
            id_elem = linksetdb.find("Link/Id")
            if id_elem is not None:
                return f"PMC{id_elem.text}"
    return None


In [20]:
def fetch_dois(pmids: list) -> dict:
    """Return a dict mapping pmid -> DOI (or None if not found)."""
    if not pmids:
        return {}
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "json",
        "tool": TOOL_NAME,
        "email": EMAIL,
    }
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    resp = requests.get(f"{EUTILS_BASE}/esummary.fcgi", params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()["result"]
    dois = {}
    for pmid in data.get("uids", []):
        rec = data[pmid]
        doi = next((a["value"] for a in rec.get("articleids", []) if a.get("idtype") == "doi"), None)
        dois[pmid] = doi
    return dois

In [21]:
def check_unpaywall(doi: str) -> dict:
    """Query Unpaywall for open-access status and the best available URL."""
    if not doi:
        return {"is_oa": False, "oa_url": None}
    resp = requests.get(f"https://api.unpaywall.org/v2/{doi}", params={"email": EMAIL}, timeout=15)
    if resp.status_code == 404:
        return {"is_oa": False, "oa_url": None}  # DOI not in Unpaywall's database
    resp.raise_for_status()
    data = resp.json()
    best_oa = data.get("best_oa_location") or {}
    return {"is_oa": data.get("is_oa", False), "oa_url": best_oa.get("url")}

In [22]:
def fetch_citation_count(doi: str) -> int | None:
    """Query Semantic Scholar for a paper's citation count, by DOI."""
    if not doi:
        return None
    url = f"https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}"
    resp = requests.get(url, params={"fields": "citationCount"}, timeout=15)
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    return resp.json().get("citationCount")

In [23]:
def select_by_either_metric(df: pd.DataFrame, relevance_col="rank", citation_col="citation_count", top_n=20):
    """Select papers ranking in the top N by *either* relevance or citation
    count, not the sum of both lists.

    Relevance rank favors recency and dense term-matching to the query, and
    can bury older, foundational papers (e.g. a landmark 2003 trial ranked
    82nd out of 288 by PubMed's relevance sort). Citation count favors
    field-wide importance instead, but is biased toward older papers, since
    citations accumulate over time. Selecting on either metric lets a paper
    qualify by being highly relevant to this specific query OR structurally
    important to the field, without requiring both.

    Papers ranking well on both metrics are only counted once (set union,
    not concatenation), so the result size is <= 2 * top_n, and will be
    smaller wherever the two rankings overlap. Papers with a missing
    citation count (not found in Semantic Scholar) cannot qualify via the
    citation path, only via relevance.
    """
    top_by_relevance = set(df.nsmallest(top_n, relevance_col)["pmid"])
    top_by_citations = set(df.nlargest(top_n, citation_col, keep="all")["pmid"])
    selected_pmids = top_by_relevance | top_by_citations
    return df[df["pmid"].isin(selected_pmids)].reset_index(drop=True)

In [24]:
def fetch_abstracts(pmids: list) -> dict:
    """Return a dict mapping pmid -> abstract text (or empty string if unavailable)."""
    if not pmids:
        return {}
    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "rettype": "abstract",
        "retmode": "xml",
        "tool": TOOL_NAME,
        "email": EMAIL,
    }
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    resp = requests.get(f"{EUTILS_BASE}/efetch.fcgi", params=params, timeout=15)
    resp.raise_for_status()
    root = ET.fromstring(resp.content)

    abstracts = {}
    for article in root.findall(".//PubmedArticle"):
        pmid_elem = article.find(".//PMID")
        pmid = pmid_elem.text if pmid_elem is not None else None
        abstract_parts = article.findall(".//AbstractText")
        text = " ".join(part.text or "" for part in abstract_parts)
        if pmid:
            abstracts[pmid] = text
    return abstracts


In [25]:
NON_PRIMARY_TYPES = {
    "Review", "Comment", "Editorial", "Letter",
    "Historical Article", "Biography", "Practice Guideline", "Congresses"
}

## Step 1: Define the queries


In [42]:
#Program-specific query
angle1_query = (
    '(TRANSEURO[tiab] OR "trans-euro"[tiab] OR '
    '("Barker RA"[Author] AND ("Parkinson Disease"[MeSH] OR Parkinson*[tiab]) '
    'AND (transplant*[tiab] OR graft*[tiab] OR "cell therapy"[tiab]))) '
    'AND ("1990/01/01"[Date - Publication] : "2014/12/31"[Date - Publication]) '
    'NOT xenotransplant*[tiab] NOT xenograft*[tiab]'
)

In [43]:
# Non-program specific (but same disease & modality) query
angle2_query = (
    '("fetal ventral mesencephalic"[tiab] OR "fetal nigral"[tiab] OR '
    '"fetal mesencephalic"[tiab] OR "fetal cell transplantation"[tiab] OR '
    '"embryonic dopamine neuron*"[tiab])'
    'AND ("Parkinson Disease"[MeSH] OR Parkinson*[tiab]) '
    'AND ("1970/01/01"[Date - Publication] : "2014/12/31"[Date - Publication])'
)

In [44]:
# Field-level mechanism, graft-induced dyskinesia
angle3_query = (
    '("graft-induced dyskinesia"[tiab] OR "graft induced dyskinesia"[tiab])'
    'AND (Parkinson*[tiab] OR transplant*[tiab])'
    'AND ("1990/01/01"[Date - Publication] : "2014/12/31"[Date - Publication])'
)

## Step 2: Run each query through the pipeline


### Angle1_query : program-specific

#### Searching articles

In [45]:
search_result = esearch(angle1_query, retmax=200)
pmids = search_result["esearchresult"]["idlist"]
print(f"Total hits after exclusion: {search_result['esearchresult']['count']}")

df = esummary(pmids)
df["rank"] = range(1, len(df) + 1)

df = enforce_date_cutoff(df, cutoff_year=2014) 


Total hits after exclusion: 42


#### Pubtype classification

In [46]:
is_review = df["pubtypes"].apply(lambda types: bool(set(types) & NON_PRIMARY_TYPES))
reviews_df = df[is_review].reset_index(drop=True)
primary_df = df[~is_review].reset_index(drop=True)

print(f"Reviews: {len(reviews_df)}")
print(f"Primary literature: {len(primary_df)}")

Reviews: 25
Primary literature: 17


In [47]:
print("=== Primary literature ===")
for _, row in primary_df.iterrows():
    print(f"{row['rank']}. {row['title']} ({row['pubdate']})")

print("\n=== Reviews ===")
for _, row in reviews_df.iterrows():
    print(f"{row['rank']}. {row['title']} ({row['pubdate']})")

=== Primary literature ===
9. Which patients with Parkinson's disease participate in clinical trials? One centre's experiences with a new cell based therapy trial (TRANSEURO). (2014)
17. Animal models of neurodegenerative diseases. (2009)
27. Graft-induced dyskinesias in Parkinson's disease: what is it all about? (2010 Aug 6)
28. Anti-amyloid compounds inhibit α-synuclein aggregation induced by protein misfolding cyclic amplification (PMCA). (2014 Apr 25)
29. Transplantation of human fetal tissue for neurodegenerative diseases: validation of a new protocol for microbiological analysis and bacterial decontamination. (2014)
30. Developing stem cell therapies for Parkinson's disease: waiting until the time is right. (2014 Nov 6)
31. Modelling of a targeted nanotherapeutic 'stroma' to deliver the cytokine LIF, or XAV939, a potent inhibitor of Wnt-β-catenin signalling, for use in human fetal dopaminergic grafts in Parkinson's disease. (2014 Oct)
33. The role of anxiety in the development of

Checking if Olanow's paper ("A double-blind controlled trial of bilateral fetal nigral transplantation in Parkinson's disease") is included

In [48]:
df[df["pmid"].str.contains("12953276", case=False, na=False)]

,pmid,title,journal,pubdate,pubtypes,rank,pub_year


#### Checking PMC full-text availability

In [49]:
pmcids = []
for pmid in df["pmid"]:
    try:
        pmcids.append(check_pmc_fulltext(pmid))
    except Exception as e:
        print(f"Error checking PMID {pmid}: {e}")
        pmcids.append(None)
    time.sleep(0.4 if not NCBI_API_KEY else 0.15)

df["pmcid"] = pmcids
df["has_pmc_fulltext"] = df["pmcid"].notna()
print(f"{df['has_pmc_fulltext'].sum()} of {len(df)} papers have PMC full text")

6 of 42 papers have PMC full text


#### Fetching DOIs

In [50]:
doi_map = fetch_dois(df["pmid"].tolist())
df["doi"] = df["pmid"].map(doi_map)
print(f"{df['doi'].notna().sum()} of {len(df)} papers have a DOI on record")

42 of 42 papers have a DOI on record


#### Checking open acesss availability

In [51]:
oa_results = []
for doi in df["doi"]:
    try:
        oa_results.append(check_unpaywall(doi))
    except Exception as e:
        print(f"Error checking DOI {doi}: {e}")
        oa_results.append({"is_oa": False, "oa_url": None})
    time.sleep(0.2)

oa_df = pd.DataFrame(oa_results)
df["unpaywall_is_oa"] = oa_df["is_oa"]
df["unpaywall_url"] = oa_df["oa_url"]

df["has_any_legitimate_access"] = df["has_pmc_fulltext"] | df["unpaywall_is_oa"]
print(f"{df['has_any_legitimate_access'].sum()} of {len(df)} accessible via PMC or Unpaywall combined")

still_inaccessible = df[~df["has_any_legitimate_access"]]
print(f"\n{len(still_inaccessible)} still with no automated open route:")
for _, row in still_inaccessible.iterrows():
    print(f"- {row['title']} ({row['pubdate']})")

19 of 42 accessible via PMC or Unpaywall combined

23 still with no automated open route:
- Cell-based therapies for Parkinson's disease. (2011 Jun)
- Stem cells and the treatment of Parkinson's disease. (2014 Oct)
- Neural grafting in Parkinson's disease Problems and possibilities. (2010)
- Stem cell therapies for Parkinson's disease: are trials just around the corner? (2014)
- The future of cell therapies in the treatment of Parkinson's disease. (2007 Oct)
- Is there a future for neural transplantation? (2004)
- The cellular repair of the brain in Parkinson's disease--past, present and future. (2004 Apr)
- The search for a curative cell therapy in Parkinson's disease. (2008 Feb 15)
- Current status of clinical trials of neural transplantation in Parkinson's disease. (2012)
- Animal models of neurodegenerative diseases. (2009)
- Fetal dopaminergic transplantation trials and the future of neural grafting in Parkinson's disease. (2013 Jan)
- Scientific and ethical issues related to stem

#### Fetching citation counts

In [52]:
citation_counts = []
for doi in df["doi"]:
    try:
        citation_counts.append(fetch_citation_count(doi))
    except Exception as e:
        print(f"Error fetching citation count for {doi}: {e}")
        citation_counts.append(None)
    time.sleep(1.0)  # Semantic Scholar's unauthenticated rate limit is stricter than NCBI's

df["citation_count"] = citation_counts
print(f"{df['citation_count'].notna().sum()} of {len(df)} papers have a citation count")

42 of 42 papers have a citation count


### Angle 2_query: non-program-specific

#### Searching papers

In [53]:
search_result = esearch(angle2_query, retmax=300)
print(f"Initial hits: {search_result['esearchresult']['count']}")

df2 = esummary(search_result["esearchresult"]["idlist"])
df2["rank"] = range(1, len(df2) + 1)

df2 = enforce_date_cutoff(df2, cutoff_year=2014)
print(f"Total hits after enforced date cutoff: {len(df2)}")


Initial hits: 290
Dropped 2 paper(s) with pub_year > 2014, despite passing the API's own date filter.
Total hits after enforced date cutoff: 288


In [54]:
df2[df2["pmid"].str.contains("12953276", case=False, na=False)]

,pmid,title,journal,pubdate,pubtypes,rank,pub_year
81,12953276,A double-blind controlled trial of bilateral fetal nigral transplantation in Parkinson's disease.,Annals of neurology,2003 Sep,"[Clinical Trial, Journal Article, Randomized Controlled Trial, Research Support, Non-U.S. Gov't, Research Support, U.S. Gov't, P.H.S.]",82,2003.0


In [55]:
df2[df2["pmid"].str.contains("11236774", case=False, na=False)]

,pmid,title,journal,pubdate,pubtypes,rank,pub_year
52,11236774,Transplantation of embryonic dopamine neurons for severe Parkinson's disease.,The New England journal of medicine,2001 Mar 8,"[Clinical Trial, Journal Article, Randomized Controlled Trial, Research Support, Non-U.S. Gov't, Research Support, U.S. Gov't, P.H.S.]",53,2001.0


#### Pubtype classification

In [56]:
is_review_2 = df2["pubtypes"].apply(lambda types: bool(set(types) & NON_PRIMARY_TYPES))
reviews_df_2  = df2[is_review_2].reset_index(drop=True)
primary_df_2  = df2[~is_review_2].reset_index(drop=True)

print(f"Reviews: {len(reviews_df_2)}")
print(f"Primary literature: {len(primary_df_2)}")

Reviews: 84
Primary literature: 204


In [57]:
print("=== Primary literature ===")
for _, row in primary_df_2.iterrows():
    print(f"{row['rank']}. {row['title']} ({row['pubdate']})")

print("\n=== Reviews ===")
for _, row in reviews_df_2.iterrows():
    print(f"{row['rank']}. {row['title']} ({row['pubdate']})")

=== Primary literature ===
47. Transplantation of fetal mesencephalic tissue in Parkinson's patients. (1994)
53. Transplantation of embryonic dopamine neurons for severe Parkinson's disease. (2001 Mar 8)
58. Ethical aspects of neural tissue transplantation. (1999 Sep)
63. Bilateral fetal nigral transplantation into the postcommissural putamen in Parkinson's disease. (1995 Sep)
68. Growth factors rescue embryonic dopamine neurons from programmed cell death. (1996 Jul)
70. Dopamine neuron stimulating actions of a GDNF propeptide. (2010 Mar 18)
71. Long-term evaluation of bilateral fetal nigral transplantation in Parkinson disease. (1999 Feb)
73. Dyskinesia after fetal cell transplantation for parkinsonism: a PET study. (2002 Nov)
76. Functional fetal nigral grafts in a patient with Parkinson's disease: chemoanatomic, ultrastructural, and metabolic studies. (1996 Jun 24)
78. Pallidal neuronal discharge in Parkinson's disease following intraputamenal fetal mesencephalic allograft. (2011 Ma

#### Checking for PMC full-text availability

In [58]:
pmcids_2 = []
for pmid in df2["pmid"]:
    try:
        pmcids_2.append(check_pmc_fulltext(pmid))
    except Exception as e:
        print(f"Error checking PMID {pmid}: {e}")
        pmcids_2.append(None)
    time.sleep(0.4 if not NCBI_API_KEY else 0.15)

df2["pmcid"] = pmcids_2
df2["has_pmc_fulltext"] = df2["pmcid"].notna()
print(f"{df2['has_pmc_fulltext'].sum()} of {len(df2)} papers have PMC full text")

40 of 288 papers have PMC full text


#### Fetching DOIs

In [59]:
doi_map_2 = fetch_dois(df2["pmid"].tolist())
df2["doi"] = df2["pmid"].map(doi_map_2)
print(f"{df2['doi'].notna().sum()} of {len(df2)} papers have a DOI on record")

253 of 288 papers have a DOI on record


#### Checking open acesss availability

In [60]:
oa_results_2 = []
for doi in df2["doi"]:
    try:
        oa_results_2.append(check_unpaywall(doi))
    except Exception as e:
        print(f"Error checking DOI {doi}: {e}")
        oa_results_2.append({"is_oa": False, "oa_url": None})
    time.sleep(0.2)

oa_df_2 = pd.DataFrame(oa_results_2)
df2["unpaywall_is_oa"] = oa_df_2["is_oa"]
df2["unpaywall_url"] = oa_df_2["oa_url"]

df2["has_any_legitimate_access"] = df2["has_pmc_fulltext"] | df2["unpaywall_is_oa"]
print(f"{df2['has_any_legitimate_access'].sum()} of {len(df2)} accessible via PMC or Unpaywall combined")

still_inaccessible_2 = df2[~df2["has_any_legitimate_access"]]
print(f"\n{len(still_inaccessible_2)} still with no automated open route:")
for _, row in still_inaccessible_2.iterrows():
    print(f"- {row['title']} ({row['pubdate']})")

90 of 288 accessible via PMC or Unpaywall combined

198 still with no automated open route:
- Are synucleinopathies prion-like disorders? (2010 Nov)
- Cell transplantation for Parkinson's disease. (2004 Jun)
- Brain imaging after neural transplantation. (2010)
- Parkinson's disease: past, present, and future. (1993 Aug)
- Fetal nigral transplantation as a therapy for Parkinson's disease. (1996 Mar)
- Cell therapy and transplantation in Parkinson's disease. (2001 Apr)
- Neurosurgery for Parkinson's disease. (2001)
- Transplants in Parkinson's disease. (1991)
- Surgical treatment of Parkinson's disease. (1997 Apr)
- PET and SPECT studies in Parkinson's disease. (1997 Apr)
- Neural transplantation in Parkinson's disease. (2000)
- Parkinson's disease and alpha synuclein: is Parkinson's disease a prion-like disorder? (2013 Jan)
- Cell transplantation for the treatment of Parkinson's disease. (2001)
- The future of cell therapies in the treatment of Parkinson's disease. (2007 Oct)
- Motor di

#### Fetching citation counts

In [61]:
citation_counts_2 = []
for doi in df2["doi"]:
    try:
        citation_counts_2.append(fetch_citation_count(doi))
    except Exception as e:
        print(f"Error fetching citation count for {doi}: {e}")
        citation_counts_2.append(None)
    time.sleep(1.0)

df2["citation_count"] = citation_counts_2
print(f"{df2['citation_count'].notna().sum()} of {len(df2)} papers have a citation count")

Error fetching citation count for 10.1111/j.1469-7580.2006.00654.x: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/DOI:10.1111/j.1469-7580.2006.00654.x?fields=citationCount
Error fetching citation count for 10.1016/s0006-8993(98)00120-6: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/DOI:10.1016/s0006-8993(98)00120-6?fields=citationCount
Error fetching citation count for 10.3171/jns.2000.92.5.0863: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/DOI:10.3171/jns.2000.92.5.0863?fields=citationCount
Error fetching citation count for 10.1111/j.1460-9568.2004.03770.x: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/DOI:10.1111/j.1460-9568.2004.03770.x?fields=citationCount
Error fetching citation count for 10.3727/000000007783464975: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/DOI:10.3727/000000007783464975?fields=citationCount
Error fetching citation c

### Angle 3: MoA and GID risk

#### Searching papers

In [62]:
search_result = esearch(angle3_query, retmax=50)
pmids = search_result["esearchresult"]["idlist"]
print(f"Total hits: {search_result['esearchresult']['count']}")

df3 = esummary(pmids)
df3["rank"] = range(1, len(df3) + 1)

df3 = enforce_date_cutoff(df3, cutoff_year=2014) 


Total hits: 20


#### Pubtype classification

In [63]:
is_review_3 = df3["pubtypes"].apply(lambda types: bool(set(types) & NON_PRIMARY_TYPES))
reviews_df_3  = df3[is_review_3].reset_index(drop=True)
primary_df_3  = df3[~is_review_3].reset_index(drop=True)

print(f"Reviews: {len(reviews_df_3)}")
print(f"Primary literature: {len(primary_df_3)}")

Reviews: 6
Primary literature: 14


In [64]:
print("=== Primary literature ===")
for _, row in primary_df_3.iterrows():
    print(f"{row['rank']}. {row['title']} ({row['pubdate']})")

print("\n=== Reviews ===")
for _, row in reviews_df_3.iterrows():
    print(f"{row['rank']}. {row['title']} ({row['pubdate']})")

=== Primary literature ===
5. Effect of levodopa priming on dopamine neuron transplant efficacy and induction of abnormal involuntary movements in parkinsonian rats. (2009 Jul 1)
7. Amphetamine-induced dyskinesia in the transplanted hemi-Parkinsonian mouse. (2012)
9. Pharmacological modulation of amphetamine-induced dyskinesia in transplanted hemi-parkinsonian rats. (2012 Oct)
10. Understanding and prevention of "therapy-" induced dyskinesias. (2012)
11. Priming for L-DOPA-induced abnormal involuntary movements increases the severity of amphetamine-induced dyskinesia in grafted rats. (2009 Sep)
12. Serotonergic and dopaminergic mechanisms in graft-induced dyskinesia in a rat model of Parkinson's disease. (2012 Sep)
13. Anatomy of Graft-induced Dyskinesias: Circuit Remodeling in the Parkinsonian Striatum. (2012 Mar 1)
14. Graft placement and uneven pattern of reinnervation in the striatum is important for development of graft-induced dyskinesia. (2006 Mar)
15. The anti-dyskinetic effect

#### Checking for full PMC availability

In [65]:
pmcids_3 = []
for pmid in df3["pmid"]:
    try:
        pmcids_3.append(check_pmc_fulltext(pmid))
    except Exception as e:
        print(f"Error checking PMID {pmid}: {e}")
        pmcids_3.append(None)
    time.sleep(0.4 if not NCBI_API_KEY else 0.15)

df3["pmcid"] = pmcids_3
df3["has_pmc_fulltext"] = df3["pmcid"].notna()
print(f"{df3['has_pmc_fulltext'].sum()} of {len(df3)} papers have PMC full text")

4 of 20 papers have PMC full text


#### Fetching DOIs

In [66]:
doi_map_3 = fetch_dois(df3["pmid"].tolist())
df3["doi"] = df3["pmid"].map(doi_map_3)
print(f"{df3['doi'].notna().sum()} of {len(df3)} papers have a DOI on record")

20 of 20 papers have a DOI on record


#### Checking open access availability

In [67]:
oa_results_3 = []
for doi in df3["doi"]:
    try:
        oa_results_3.append(check_unpaywall(doi))
    except Exception as e:
        print(f"Error checking DOI {doi}: {e}")
        oa_results_3.append({"is_oa": False, "oa_url": None})
    time.sleep(0.2)

oa_df_3 = pd.DataFrame(oa_results_3)
df3["unpaywall_is_oa"] = oa_df_3["is_oa"]
df3["unpaywall_url"] = oa_df_3["oa_url"]

df3["has_any_legitimate_access"] = df3["has_pmc_fulltext"] | df3["unpaywall_is_oa"]
print(f"{df3['has_any_legitimate_access'].sum()} of {len(df3)} accessible via PMC or Unpaywall combined")

still_inaccessible_3 = df3[~df3["has_any_legitimate_access"]]
print(f"\n{len(still_inaccessible_3)} still with no automated open route:")
for _, row in still_inaccessible_3.iterrows():
    print(f"- {row['title']} ({row['pubdate']})")

11 of 20 accessible via PMC or Unpaywall combined

9 still with no automated open route:
- Understanding graft-induced dyskinesia. (2010 Sep)
- L-DOPA- and graft-induced dyskinesia following transplantation. (2012)
- L-DOPA and graft-induced dyskinesia: different treatment, same story? (2013 Jul)
- Clinical and experimental experiences of graft-induced dyskinesia. (2011)
- Neural grafting in Parkinson's disease unraveling the mechanisms underlying graft-induced dyskinesia. (2010)
- The future of cell therapies in the treatment of Parkinson's disease. (2007 Oct)
- Priming for L-DOPA-induced abnormal involuntary movements increases the severity of amphetamine-induced dyskinesia in grafted rats. (2009 Sep)
- Extent of pre-operative L-DOPA-induced dyskinesia predicts the severity of graft-induced dyskinesia after fetal dopamine cell transplantation. (2011 Dec)
- Impact of dopamine versus serotonin cell transplantation for the development of graft-induced dyskinesia in a rat Parkinson model

#### Fetching citation counts

In [68]:
citation_counts_3 = []
for doi in df3["doi"]:
    try:
        citation_counts_3.append(fetch_citation_count(doi))
    except Exception as e:
        print(f"Error fetching citation count for {doi}: {e}")
        citation_counts_3.append(None)
    time.sleep(1.0)

df3["citation_count"] = citation_counts_3
print(f"{df3['citation_count'].notna().sum()} of {len(df3)} papers have a citation count")

20 of 20 papers have a citation count


## Step 3: Building final corpus (with cutoff)


#### Select top_n papers from each query by union of relevance and citations
- A fixed cutoff was chosen for each individual ranking - relevance AND citations. Namely: top_n=10 papers for both angle1 and angle2 searches; and top_n=5 for the narrower angle3.


In [69]:
df_selected = select_by_either_metric(df, top_n=10)  # arbitrary cut-off of 10 papers 
df2_selected = select_by_either_metric(df2, top_n=10) # arbitrary cut-off of 10 papers
df3_selected = select_by_either_metric(df3, top_n=5) # arbitrary cut-off of 5 papers

print(f"Angle1 selected: {len(df_selected)}, Angle2 selected: {len(df2_selected)}, Angle3 selected: {len(df3_selected)}")



Angle1 selected: 16, Angle2 selected: 20, Angle3 selected: 10


In [68]:
df_selected

,pmid,title,journal,pubdate,pubtypes,rank,pub_year,pmcid,has_pmc_fulltext,doi,unpaywall_is_oa,unpaywall_url,has_any_legitimate_access,citation_count
0,21651331,Cell-based therapies for Parkinson's disease.,Expert review of neurotherapeutics,2011 Jun,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",1,2011.0,None,False,10.1586/ern.11.33,False,None,False,54
1,20489615,Cell transplantation in Parkinson's disease: problems and perspectives.,Current opinion in neurology,2010 Aug,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",2,2010.0,None,False,10.1097/WCO.0b013e32833b1f62,True,https://zenodo.org/record/3429361,True,65
2,23298521,Stem cells and the treatment of Parkinson's disease.,Experimental neurology,2014 Oct,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",3,2014.0,None,False,10.1016/j.expneurol.2012.12.017,False,None,False,28
3,19007882,Cell replacement therapy for Parkinson's disease.,Biochimica et biophysica acta,2009 Jul,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",4,2009.0,None,False,10.1016/j.bbadis.2008.10.007,True,https://hal.science/hal-00492064,True,83
4,30890881,Stem cells and regenerative therapies for Parkinson's disease.,Degenerative neurological and neuromuscular disease,2012,"[Journal Article, Review]",5,2012.0,PMC6065567,True,10.2147/DNND.S16087,True,https://www.dovepress.com/getfile.php?fileID=13337,True,4
5,20887880,Neural grafting in Parkinson's disease Problems and possibilities.,Progress in brain research,2010,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",6,2010.0,None,False,10.1016/S0079-6123(10)84014-2,False,None,False,147
6,25372072,Stem cell therapies for Parkinson's disease: are trials just around the corner?,Regenerative medicine,2014,"[Editorial, Research Support, Non-U.S. Gov't]",7,2014.0,None,False,10.2217/rme.14.43,False,None,False,9
7,17916042,The future of cell therapies in the treatment of Parkinson's disease.,Expert opinion on biological therapy,2007 Oct,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",8,2007.0,None,False,10.1517/14712598.7.10.1487,False,None,False,17
8,25170676,Which patients with Parkinson's disease participate in clinical trials? One centre's experiences with a new cell based therapy trial (TRANSEURO).,Journal of Parkinson's disease,2014,"[Journal Article, Multicenter Study, Randomized Controlled Trial, Research Support, Non-U.S. Gov't]",9,2014.0,None,False,10.3233/JPD-140432,True,https://doaj.org/article/efe6ca6432c043b790ad6c10182c2930,True,47
9,24372386,The future of cell therapies and brain repair: Parkinson's disease leads the way.,Neuropathology and applied neurobiology,2014 Feb,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",10,2014.0,None,False,10.1111/nan.12110,True,https://doi.org/10.1111/nan.12110,True,83


In [69]:
df2_selected

,pmid,title,journal,pubdate,pubtypes,rank,pub_year,pmcid,has_pmc_fulltext,doi,unpaywall_is_oa,unpaywall_url,has_any_legitimate_access,citation_count
0,21901584,Cell therapeutics in Parkinson's disease.,Neurotherapeutics : the journal of the American Society for Experimental NeuroTherapeutics,2011 Oct,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",1,2011.0,PMC3250287,True,10.1007/s13311-011-0069-6,True,https://link.springer.com/content/pdf/10.1007/s13311-011-0069-6.pdf,True,91.0
1,15717042,Cell therapy in Parkinson's disease.,NeuroRx : the journal of the American Society for Experimental NeuroTherapeutics,2004 Oct,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",2,2004.0,PMC534947,True,10.1602/neurorx.1.4.382,True,https://link.springer.com/content/pdf/10.1602%2Fneurorx.1.4.382.pdf,True,124.0
2,7975571,Treatment of Parkinson's disease.,The Western journal of medicine,1994 Sep,"[Journal Article, Review]",3,1994.0,PMC1011414,True,None,False,None,True,NaN
3,21626551,"Parkinson's disease, proteins, and prions: milestones.",Movement disorders : official journal of the Movement Disorder Society,2011 May,"[Historical Article, Journal Article, Review]",4,2011.0,None,False,10.1002/mds.23767,True,https://onlinelibrary.wiley.com/doi/pdfdirect/10.1002/mds.23767,True,51.0
4,20846907,Are synucleinopathies prion-like disorders?,The Lancet. Neurology,2010 Nov,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",5,2010.0,None,False,10.1016/S1474-4422(10)70213-1,False,None,False,266.0
5,22936307,α-Synuclein and neuronal cell death.,Molecular neurobiology,2013 Apr,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",6,2013.0,PMC3589663,True,10.1007/s12035-012-8327-0,True,https://link.springer.com/content/pdf/10.1007/s12035-012-8327-0.pdf,True,0.0
6,15198860,Cell transplantation for Parkinson's disease.,Neurological research,2004 Jun,"[Journal Article, Review]",7,2004.0,None,False,10.1179/016164104225017604,False,None,False,NaN
7,20887876,Brain imaging after neural transplantation.,Progress in brain research,2010,"[Journal Article, Review]",8,2010.0,None,False,10.1016/S0079-6123(10)84010-5,False,None,False,23.0
8,23401015,Developing dopaminergic cell therapy for Parkinson's disease--give up or move forward?,Movement disorders : official journal of the Movement Disorder Society,2013 Mar,"[Editorial, Review]",9,2013.0,None,False,10.1002/mds.25378,True,https://onlinelibrary.wiley.com/doi/pdfdirect/10.1002/mds.25378,True,78.0
9,8397719,"Parkinson's disease: past, present, and future.",Neuropsychopharmacology : official publication of the American College of Neuropsychopharmacology,1993 Aug,"[Historical Article, Journal Article, Review]",10,1993.0,None,False,10.1038/npp.1993.39,False,None,False,44.0


In [71]:
df3_selected

,pmid,title,journal,pubdate,pubtypes,rank,pub_year,pmcid,has_pmc_fulltext,doi,unpaywall_is_oa,unpaywall_url,has_any_legitimate_access,citation_count
0,20868333,Understanding graft-induced dyskinesia.,Regenerative medicine,2010 Sep,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",1,2010.0,None,False,10.2217/rme.10.42,False,None,False,16
1,23195418,L-DOPA- and graft-induced dyskinesia following transplantation.,Progress in brain research,2012,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",2,2012.0,None,False,10.1016/B978-0-444-59575-1.00007-7,False,None,False,14
2,23828589,"L-DOPA and graft-induced dyskinesia: different treatment, same story?","Experimental biology and medicine (Maywood, N.J.)",2013 Jul,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",3,2013.0,None,False,10.1177/1535370213488478,False,None,False,5
3,21907087,Clinical and experimental experiences of graft-induced dyskinesia.,International review of neurobiology,2011,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",4,2011.0,None,False,10.1016/B978-0-12-381328-2.00007-9,False,None,False,9
4,19399877,Effect of levodopa priming on dopamine neuron transplant efficacy and induction of abnormal involuntary movements in parkinsonian rats.,The Journal of comparative neurology,2009 Jul 1,"[Journal Article, Research Support, N.I.H., Extramural]",5,2009.0,PMC2886671,True,10.1002/cne.22037,True,https://www.ncbi.nlm.nih.gov/pmc/articles/2886671,True,40
5,20887881,Neural grafting in Parkinson's disease unraveling the mechanisms underlying graft-induced dyskinesia.,Progress in brain research,2010,"[Journal Article, Research Support, Non-U.S. Gov't, Review]",6,2010.0,None,False,10.1016/S0079-6123(10)84015-4,False,None,False,78
6,19393238,Priming for L-DOPA-induced abnormal involuntary movements increases the severity of amphetamine-induced dyskinesia in grafted rats.,Experimental neurology,2009 Sep,"[Journal Article, Research Support, Non-U.S. Gov't]",11,2009.0,None,False,10.1016/j.expneurol.2009.04.010,False,None,False,51
7,22579773,Serotonergic and dopaminergic mechanisms in graft-induced dyskinesia in a rat model of Parkinson's disease.,Neurobiology of disease,2012 Sep,"[Journal Article, Research Support, Non-U.S. Gov't, Video-Audio Media]",12,2012.0,None,False,10.1016/j.nbd.2012.03.038,True,https://doaj.org/article/1e207e4e620e4951aa04424fd03c5cff,True,42
8,16256359,Graft placement and uneven pattern of reinnervation in the striatum is important for development of graft-induced dyskinesia.,Neurobiology of disease,2006 Mar,"[Journal Article, Research Support, Non-U.S. Gov't]",14,2006.0,None,False,10.1016/j.nbd.2005.09.008,True,https://doaj.org/article/d9298519bc2147b29208360ae35020fa,True,121
9,16406222,The impact of graft size on the development of dyskinesia following intrastriatal grafting of embryonic dopamine neurons in the rat.,Neurobiology of disease,2006 May,"[Journal Article, Research Support, Non-U.S. Gov't]",19,2006.0,None,False,10.1016/j.nbd.2005.11.011,True,https://doaj.org/article/5a41a29fec994d34a44f87068c26e1e0,True,92


#### Corpus: combine selected papers from all queries and remove duplicates

In [73]:
combined = pd.concat(
    [df_selected.assign(query_angle="angle1"), df2_selected.assign(query_angle="angle2"), df3_selected.assign(query_angle="angle3")],
    ignore_index=True
)

In [74]:
corpus = (
    combined
    .groupby("pmid", as_index=False)
    .agg({
        "title": "first", "journal": "first", "pubdate": "first",
        "pubtypes": "first", "pmcid": "first", "has_pmc_fulltext": "first",
        "doi": "first", "unpaywall_is_oa": "first", "unpaywall_url": "first",
        "has_any_legitimate_access": "first",
        "query_angle": lambda x: sorted(set(x)),
    })
)

print(f"Combined before dedup: {len(combined)}, after dedup: {len(corpus)}")

Combined before dedup: 46, after dedup: 46


**Confirm trial design paper (Moore, 2014) is included in corpus**

In [75]:
corpus[corpus["pmid"].isin(["25170676"])]

,pmid,title,journal,pubdate,pubtypes,pmcid,has_pmc_fulltext,doi,unpaywall_is_oa,unpaywall_url,has_any_legitimate_access,query_angle
35,25170676,Which patients with Parkinson's disease participate in clinical trials? One centre's experiences with a new cell based therapy trial (TRANSEURO).,Journal of Parkinson's disease,2014,"[Journal Article, Multicenter Study, Randomized Controlled Trial, Research Support, Non-U.S. Gov't]",None,False,10.3233/JPD-140432,True,https://doaj.org/article/efe6ca6432c043b790ad6c10182c2930,True,[angle1]


**Confirm Freed (PMID 11236774) and Olanow (PMID 12953276) is included in corpus**

In [76]:
corpus[corpus["pmid"].isin(["11236774", "12953276"])]

,pmid,title,journal,pubdate,pubtypes,pmcid,has_pmc_fulltext,doi,unpaywall_is_oa,unpaywall_url,has_any_legitimate_access,query_angle
0,11236774,Transplantation of embryonic dopamine neurons for severe Parkinson's disease.,The New England journal of medicine,2001 Mar 8,"[Clinical Trial, Journal Article, Randomized Controlled Trial, Research Support, Non-U.S. Gov't, Research Support, U.S. Gov't, P.H.S.]",None,False,10.1056/NEJM200103083441002,True,https://www.nejm.org/doi/pdf/10.1056/NEJM200103083441002?articleTools=true,True,[angle2]
2,12953276,A double-blind controlled trial of bilateral fetal nigral transplantation in Parkinson's disease.,Annals of neurology,2003 Sep,"[Clinical Trial, Journal Article, Randomized Controlled Trial, Research Support, Non-U.S. Gov't, Research Support, U.S. Gov't, P.H.S.]",None,False,10.1002/ana.10720,True,https://onlinelibrary.wiley.com/doi/pdfdirect/10.1002/ana.10720,True,[angle2]


**Check if Politis et al 2010 (PMID 20592420) - serotoninergic neurons as cause of GIDs -  is included in corpus**

In [77]:
corpus[corpus["pmid"].isin(["20592420"])]

,pmid,title,journal,pubdate,pubtypes,pmcid,has_pmc_fulltext,doi,unpaywall_is_oa,unpaywall_url,has_any_legitimate_access,query_angle


#### Save corpus list as CSV

In [80]:
corpus.to_csv("transeuro_corpus_pre2015.csv", index=False)
print("Saved to transeuro_corpus_pre2015.csv")

Saved to transeuro_corpus_pre2015.csv


#### Fetch abstracts of finalised corpus

In [81]:
final_pmids = corpus["pmid"].tolist()
abstract_map = fetch_abstracts(final_pmids)
corpus["abstract"] = corpus["pmid"].map(abstract_map)
print(f"{corpus['abstract'].str.len().gt(0).sum()} of {len(corpus)} papers have abstract text")

45 of 46 papers have abstract text


## Step 4: Build frozen context

In [40]:
def build_frozen_context(corpus: pd.DataFrame) -> str:
    """Format the final corpus into one frozen text block for Week 1/2 prompts.
    Ordered chronologically (oldest first), a neutral rule not based on
    judged importance. Each entry is tagged by PMID for traceable citation."""
    ordered = corpus.copy()
    ordered["pub_year_sort"] = ordered["pubdate"].str.extract(r"(\d{4})").astype(float)
    ordered = ordered.sort_values("pub_year_sort").reset_index(drop=True)

    entries = []
    for _, row in ordered.iterrows():
        abstract_text = row["abstract"].strip() if isinstance(row["abstract"], str) else ""
        if not abstract_text:
            abstract_text = "[No abstract available through automated retrieval]"
        entries.append(
            f"[PMID: {row['pmid']}]\n"
            f"Title: {row['title']}\n"
            f"Journal: {row['journal']} ({row['pubdate']})\n"
            f"Abstract: {abstract_text}\n"
        )

    header = (
        f"EVIDENCE PACKAGE — {len(ordered)} sources, all published on or before "
        f"2014-12-31. This is the complete evidence base available for this "
        f"assessment. When referencing a specific claim, cite the source's "
        f"PMID.\n\n"
    )
    return header + "\n---\n".join(entries)


In [82]:
frozen_context = build_frozen_context(corpus)
print(f"Frozen context: {len(frozen_context)} characters (~{len(frozen_context) // 4} tokens)")

with open("transeuro_frozen_context.txt", "w", encoding="utf-8") as f:
    f.write(frozen_context)
print("Saved to transeuro_frozen_context.txt")

Frozen context: 74593 characters (~18648 tokens)
Saved to transeuro_frozen_context.txt


## Pre-LLM step: Loading frozen context and declaring Anthropic API

In [9]:
with open("transeuro_frozen_context.txt", "r", encoding="utf-8") as f:
    frozen_context = f.read()

In [12]:
load_dotenv()
client = Anthropic()
MODEL = "claude-sonnet-5"

## Step 5: Provide frozen context to multi-persona LLM

In [87]:
week1_prompt = f"""You are reviewing a therapeutic development decision as of December 31, 2014.

DECISION: TRANSEURO, a European trial redesigning fetal ventral mesencephalic
tissue transplantation for Parkinson's disease, is about to begin grafting
patients. Assess whether this program is well-justified to proceed, given
everything known up to this date, including the graft-induced dyskinesia
problem observed in earlier (1990s-2000s) NIH-funded trials.

You must reason only from the evidence package below. If you find yourself
drawing on knowledge not contained in this package, say so explicitly rather
than presenting it as if it came from the evidence.

Respond as four independent reviewers, each in their own clearly labeled
section. Each should state their top three concerns and their confidence in
each, citing evidence by PMID.

1. BIOLOGY — success criterion: does the mechanistic rationale for this
   redesign adequately address the known cause of graft-induced dyskinesia?
2. CLINICAL — success criterion: is the trial design and patient selection
   adequate given prior trials' safety and efficacy findings?
3. REGULATORY — success criterion: are known risks adequately characterized
   and mitigated for ethical/regulatory approval to proceed?
4. CMC/MANUFACTURING — success criterion: is fetal tissue sourcing and
   preparation feasible at the scale this trial requires?


EVIDENCE PACKAGE:
{frozen_context}
"""

response = client.messages.create(
    model=MODEL,
    max_tokens=16000,
    temperature=1,
    thinking={
        "type": "adaptive",
        "display": "summarized"
    },
    output_config={
        "effort": "high"
    },
    messages=[{"role": "user", "content": week1_prompt}],
)

week1_output = "".join(
    block.text for block in response.content if block.type == "text"
)

print(week1_output)


with open("week1_output_run2_with_angle3.txt", "w", encoding="utf-8") as f:
    f.write(week1_output)



**Note on method:** The following assessments are constructed strictly from the 46 sources supplied. Where a reviewer's reasoning would normally draw on outside knowledge (e.g., specific TRANSEURO protocol details, exact TRANSEURO patient-age cutoffs, GDNF trial outcomes, iPSC clinical timelines), I flag this explicitly as "not established in this package" rather than asserting it.

---

## 1. BIOLOGY REVIEWER

**Top concern 1 — No single confirmed mechanism for graft-induced dyskinesia (GID).**
The package contains at least five competing, only partially overlapping hypotheses for GID: uneven/patchy striatal reinnervation from graft placement (PMID 16256359), graft size (PMID 16406222), serotonergic neuron contamination of the graft (PMID 22579773), pre-graft L-DOPA priming (PMID 19399877, 19393238), and host inflammatory/immune reaction (PMID 15717048). Review papers as late as 2011–2012 state plainly that "questions still remain as to the underlying mechanism" (PMID 21907087) and th

#### Record and save thinking process

In [88]:
week1_thinking = "".join(block.thinking for block in response.content if block.type == "thinking")
print(week1_thinking)

with open("week1_thinking_run2_with_angle3.txt", "w", encoding="utf-8") as f:
    f.write(week1_thinking)

Looking at the clinical evidence systematically, the early open-label trials from the late 1980s and early 1990s showed modest benefits with demonstrated graft survival, but the pivotal double-blind controlled trials in 2001 and 2003 failed their primary endpoints while uncovering a critical problem—graft-induced dyskinesia emerged as a major adverse effect, with one study showing 15% dyskinesia recurrence and another revealing 56% of patients developed off-medication dyskinesia. Now I'm examining the mechanistic work to understand what's driving these outcomes.

The research points to several mechanisms behind graft-induced dyskinesia: uneven reinnervation patterns, graft size effects, serotonergic contamination from the tissue, and L-DOPA priming effects. Multiple review articles have synthesized these competing hypotheses, and there's concerning evidence that Lewy body pathology can propagate from the host brain into the graft—a long-term disease progression risk that wasn't initial

## Step 6 : Provide frozen context to LLM through independent persona calls 

In [4]:
PERSONAS = {
    "biology": "does the mechanistic rationale for this redesign adequately address the known cause of graft-induced dyskinesia?",
    "clinical": "is the trial design and patient selection adequate given prior trials' safety and efficacy findings?",
    "regulatory": "are known risks adequately characterized and mitigated for ethical/regulatory approval to proceed?",
    "cmc_manufacturing": "is fetal tissue sourcing and preparation feasible at the scale this trial requires?",
}



In [89]:
week2_outputs = {}

for persona_name, success_criterion in PERSONAS.items():
    persona_prompt = f"""You are reviewing a therapeutic development decision as of December 31, 2014,
from the perspective of a single specialist: {persona_name.upper()}.

DECISION: TRANSEURO, a European trial redesigning fetal ventral mesencephalic
tissue transplantation for Parkinson's disease, is about to begin grafting
patients. Assess whether this program is well-justified to proceed, given
everything known up to this date, including the graft-induced dyskinesia
problem observed in earlier (1990s-2000s) NIH-funded trials.

You must reason only from the evidence package below. If you find yourself
drawing on knowledge not contained in this package, say so explicitly rather
than presenting it as if it came from the evidence.

Your specific success criterion: {success_criterion}

State your top three concerns and your confidence in each, citing evidence by PMID.

EVIDENCE PACKAGE:
{frozen_context}
"""

    response = client.messages.create(
        model=MODEL,
        max_tokens=16000,
        temperature=1,
        thinking={"type": "adaptive", "display": "summarized"},
        output_config={"effort": "high"},
        messages=[{"role": "user", "content": persona_prompt}],
    )

    text = "".join(b.text for b in response.content if b.type == "text")
    thinking = "".join(b.thinking for b in response.content if b.type == "thinking")

    print(f"{persona_name}: stop_reason={response.stop_reason}, "
          f"text={len(text)} chars, thinking={len(thinking)} chars")

    week2_outputs[persona_name] = text

    with open(f"week2_{persona_name}_run2_with_angle3.txt", "w", encoding="utf-8") as f:
        f.write(text)
    with open(f"week2_{persona_name}_run2_with_angle3_thinking.txt", "w", encoding="utf-8") as f:
        f.write(thinking)

    time.sleep(1.0)  # light pacing between calls, not strictly required but polite

biology: stop_reason=end_turn, text=6053 chars, thinking=8340 chars
clinical: stop_reason=end_turn, text=6643 chars, thinking=5025 chars
regulatory: stop_reason=end_turn, text=5872 chars, thinking=4738 chars
cmc_manufacturing: stop_reason=end_turn, text=6414 chars, thinking=4217 chars


## Step 7: Provide frozen context to multi-persona agents with one-call tool

In [14]:
def date_capped_pubmed_search(query_term: str, max_results: int = 5) -> str:
    """Date-capped PubMed search for agent use. The cutoff is fixed and
    cannot be altered by the agent's query. Enforced twice: once in the
    query itself, once by re-checking the actual returned pubdate, since
    the query-time filter alone has previously let post-cutoff results through."""
    CUTOFF_YEAR = 2014
    capped_query = f'({query_term}) AND ("1900/01/01"[pdat] : "{CUTOFF_YEAR}/12/31"[pdat])'

    search_result = esearch(capped_query, retmax=max_results, sort="relevance")
    pmids = search_result["esearchresult"]["idlist"]
    if not pmids:
        return "No results found within the date-capped search."

    df = esummary(pmids)
    df["pub_year"] = df["pubdate"].str.extract(r"(\d{4})").astype(float)
    before = len(df)
    df = df[df["pub_year"] <= CUTOFF_YEAR]
    if before - len(df) > 0:
        print(f"[tool enforcement] dropped {before - len(df)} result(s) failing strict date recheck")

    abstracts = fetch_abstracts(df["pmid"].tolist())
    entries = [
        f"[PMID: {row['pmid']}] {row['title']} ({row['pubdate']})\n"
        f"{abstracts.get(row['pmid'], '[abstract unavailable]')}"
        for _, row in df.iterrows()
    ]
    return "\n\n---\n\n".join(entries) if entries else "No results after date enforcement."


PUBMED_TOOL_SCHEMA = {
    "name": "search_pubmed",
    "description": (
        "Search PubMed for relevant literature. Results are automatically "
        "restricted to sources published on or before 2014-12-31; this "
        "restriction cannot be changed or bypassed."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query_term": {"type": "string", "description": "PubMed search query"},
            "max_results": {"type": "integer", "description": "Number of results (default 5)"},
        },
        "required": ["query_term"],
    },
}

In [38]:
def run_week3_persona(persona_name: str, success_criterion: str):
    prompt = f"""You are reviewing a therapeutic development decision as of December 31, 2014,
from the perspective of a single specialist: {persona_name.upper()}.

DECISION: TRANSEURO, a European trial redesigning fetal ventral mesencephalic
tissue transplantation for Parkinson's disease, is about to begin grafting
patients. Assess whether this program is well-justified to proceed.

Your specific success criterion: {success_criterion}

EVIDENCE PACKAGE:
{frozen_context}

You also have access to a PubMed search tool, restricted to sources published
on or before 2014-12-31, if you judge the evidence package above to be
insufficient for a well-supported assessment on any point. 

State your top three concerns and your confidence in each, citing evidence
by PMID from either source."""

    messages = [{"role": "user", "content": prompt}]

    response = client.messages.create(
        model=MODEL, max_tokens=16000, tools=[PUBMED_TOOL_SCHEMA],
        thinking={"type": "adaptive", "display": "summarized"},
        output_config={"effort": "high"}, messages=messages,
    )

    tool_called = any(b.type == "tool_use" for b in response.content)

    if tool_called:
        messages.append({"role": "assistant", "content": response.content})
        tool_results = [
            {"type": "tool_result", "tool_use_id": b.id,
             "content": date_capped_pubmed_search(**b.input)[:8000]}
            for b in response.content if b.type == "tool_use" and b.name == "search_pubmed"
        ]
        tool_results.append({
            "type": "text",
            "text": ("Note: this was your one available tool call for this assessment. "
                     "No further searches are possible. Based on the evidence above and "
                     "the original evidence package, provide your complete final "
                     "assessment now.")
        })
        messages.append({"role": "user", "content": tool_results})

        # No `tools` param here at all — a second call is structurally impossible
        response = client.messages.create(
            model=MODEL, max_tokens=16000,
            thinking={"type": "adaptive", "display": "summarized"},
            output_config={"effort": "high"}, messages=messages,
        )

    text = "".join(b.text for b in response.content if b.type == "text")
    thinking = "".join(b.thinking for b in response.content if b.type == "thinking")
    
    if len(text) == 0:
        print(f"[DIAGNOSTIC] {persona_name}: empty text, dumping response.content structure")
        print([b.type for b in response.content])
        for b in response.content:
            if b.type == "thinking":
                print(f"  thinking (first 300 chars): {b.thinking[:300]}")
            elif b.type == "tool_use":
                print(f"  tool_use: {b.input}")
        with open(f"week3_{persona_name}_DEBUG_raw_response.txt", "w", encoding="utf-8") as f:
            f.write(str(response.content))

    return text, thinking, response.stop_reason, tool_called

In [37]:
def run_week3_persona_with_retry(persona_name: str, success_criterion: str, max_attempts: int = 3):
    for attempt in range(1, max_attempts + 1):
        text, thinking, stop_reason, tool_called = run_week3_persona(persona_name, success_criterion)
        if len(text) > 0:
            return text, thinking, stop_reason, tool_called
        print(f"{persona_name}: empty text on attempt {attempt}, retrying...")
    print(f"{persona_name}: still empty after {max_attempts} attempts, keeping last result")
    return text, thinking, stop_reason, tool_called

In [44]:
week3_outputs = {}
for persona_name, success_criterion in PERSONAS.items():
    try:
        text, thinking, stop_reason, tool_called = run_week3_persona_with_retry(persona_name, success_criterion)
        print(f"{persona_name}: stop_reason={stop_reason}, tool_called={tool_called}, text={len(text)} chars")
        week3_outputs[persona_name] = text
        with open(f"week3_{persona_name}_run4.txt", "w", encoding="utf-8") as f:
            f.write(text)
        with open(f"week3_{persona_name}_run4_thinking.txt", "w", encoding="utf-8") as f:
            f.write(thinking)
    except Exception as e:
        print(f"{persona_name}: FAILED — {e}")
    time.sleep(1.0)

biology: stop_reason=end_turn, tool_called=True, text=5406 chars
clinical: stop_reason=end_turn, tool_called=True, text=5842 chars
regulatory: stop_reason=end_turn, tool_called=True, text=6544 chars
cmc_manufacturing: stop_reason=end_turn, tool_called=True, text=6198 chars


## Step 8: Provide frozen context to multi-persona agents with multi-call tool 

In [40]:
MAX_TOOL_ROUNDS = 10  # a safety valve

def run_week4_persona(persona_name: str, success_criterion: str):
    prompt = f"""You are reviewing a therapeutic development decision as of December 31, 2014,
from the perspective of a single specialist: {persona_name.upper()}.

DECISION: TRANSEURO, a European trial redesigning fetal ventral mesencephalic
tissue transplantation for Parkinson's disease, is about to begin grafting
patients. Assess whether this program is well-justified to proceed.

Your specific success criterion: {success_criterion}

EVIDENCE PACKAGE:
{frozen_context}

You also have access to a PubMed search tool, restricted to sources published
on or before 2014-12-31, if you judge the evidence package above to be
insufficient for a well-supported assessment on any point. 

State your top three concerns and your confidence in each, citing evidence
by PMID from either source."""

    messages = [{"role": "user", "content": prompt}]
    all_tool_calls = []
    all_thinking = []
    rounds = 0

    response = client.messages.create(
        model=MODEL, max_tokens=16000, tools=[PUBMED_TOOL_SCHEMA],
        thinking={"type": "adaptive", "display": "summarized"},
        output_config={"effort": "high"}, messages=messages,
    )
    all_thinking.append("".join(b.thinking for b in response.content if b.type == "thinking"))

    while response.stop_reason == "tool_use" and rounds < MAX_TOOL_ROUNDS:
        rounds += 1
        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for b in response.content:
            if b.type == "tool_use" and b.name == "search_pubmed":
                all_tool_calls.append(b.input)
                result = date_capped_pubmed_search(**b.input)
                tool_results.append({
                    "type": "tool_result", "tool_use_id": b.id, "content": result[:8000]
                })
        messages.append({"role": "user", "content": tool_results})

        response = client.messages.create(
            model=MODEL, max_tokens=16000, tools=[PUBMED_TOOL_SCHEMA],
            thinking={"type": "adaptive", "display": "summarized"},
            output_config={"effort": "high"}, messages=messages,
        )
        all_thinking.append("".join(b.thinking for b in response.content if b.type == "thinking"))

    if rounds == MAX_TOOL_ROUNDS and response.stop_reason == "tool_use":
        print(f"[WARNING] {persona_name} hit the {MAX_TOOL_ROUNDS}-round safety cap, "
              f"still requesting tools. This is a real finding worth noting, not just a limit to raise.")

    text = "".join(b.text for b in response.content if b.type == "text")
    return text, "\n---\n".join(all_thinking), response.stop_reason, all_tool_calls, rounds

In [45]:
week4_outputs = {}
for persona_name, success_criterion in PERSONAS.items():
    try:
        text, thinking, stop_reason, tool_calls, rounds = run_week4_persona(persona_name, success_criterion)
        print(f"{persona_name}: stop_reason={stop_reason}, rounds={rounds}, "
              f"tool_calls={tool_calls}, text={len(text)} chars")
        week4_outputs[persona_name] = text
        with open(f"week4_{persona_name}_run4.txt", "w", encoding="utf-8") as f:
            f.write(text)
        with open(f"week4_{persona_name}_run4_thinking.txt", "w", encoding="utf-8") as f:
            f.write(thinking)
    except Exception as e:
        print(f"{persona_name}: FAILED — {e}")
    time.sleep(1.0)

biology: stop_reason=end_turn, rounds=1, tool_calls=[{'query_term': 'TRANSEURO fetal ventral mesencephalic tissue dissection serotonin dopamine ratio graft-induced dyskinesia', 'max_results': 6}, {'query_term': 'graft-induced dyskinesia mechanism unresolved rostral caudal putamen serotonin', 'max_results': 6}], text=6019 chars
clinical: stop_reason=end_turn, rounds=1, tool_calls=[{'query_term': 'TRANSEURO trial protocol fetal cell transplantation Parkinson patient selection criteria', 'max_results': 5}, {'query_term': 'graft-induced dyskinesia clinical predictors Parkinson fetal transplant patient selection age severity', 'max_results': 5}], text=5422 chars
regulatory: stop_reason=end_turn, rounds=1, tool_calls=[{'query_term': 'TRANSEURO trial design fetal ventral mesencephalic transplantation Parkinson', 'max_results': 5}, {'query_term': 'fetal tissue transplantation Parkinson disease patient selection graft-induced dyskinesia prevention protocol', 'max_results': 5}], text=6273 chars
